# Notebook 02: Data Cleaning & Merging

This notebook loads all raw data from Notebook 01, cleans it, and merges into a
master dataframe with engineered features.

**Output:** `data/processed/master.csv` (379 rows, 24 columns)

In [4]:
# Cell 1: Load raw data
import pandas as pd
import numpy as np
import os

RAW = '../data/raw'
PROCESSED = '../data/processed'
os.makedirs(PROCESSED, exist_ok=True)

posts = pd.read_csv(os.path.join(RAW, 'truth_posts.csv'))
posts['timestamp'] = pd.to_datetime(posts['timestamp'], errors='coerce')
posts = posts.dropna(subset=['timestamp'])
posts['date'] = pd.to_datetime(posts['timestamp'].dt.date)

# FRED CSVs use observation_date + series name as column headers
wti = pd.read_csv(os.path.join(RAW, 'wti_daily.csv'))
wti.columns = ['date', 'wti_close']
wti['date'] = pd.to_datetime(wti['date'])
wti['wti_close'] = pd.to_numeric(wti['wti_close'], errors='coerce')
wti = wti.dropna(subset=['wti_close'])

brent = pd.read_csv(os.path.join(RAW, 'brent_daily.csv'))
brent.columns = ['date', 'brent_close']
brent['date'] = pd.to_datetime(brent['date'])
brent['brent_close'] = pd.to_numeric(brent['brent_close'], errors='coerce')
brent = brent.dropna(subset=['brent_close'])

print(f'Posts: {len(posts)} total, Iran-related: {posts["iran_related"].sum()}')
print(f'WTI: {len(wti)} days | Brent: {len(brent)} days')

Posts: 21013 total, Iran-related: 2695
WTI: 365 days | Brent: 374 days


In [5]:
# Cell 2: Clean post data — classify direction, tag fabrication
IRAN_KEYWORDS = [
    'iran', 'iranian', 'tehran', 'khamenei', 'persian', 'strait of hormuz',
    'nuclear deal', 'sanctions', 'middle east', 'oil', 'crude', 'opec',
    'bomb', 'strike', 'military', 'war', 'troops', 'attack', 'missile',
    'navy', 'gulf', 'escalat', 'de-escalat', 'peace talk', 'negotiate',
    'surrender', 'ceasefire', 'regime', 'threat'
]

ESCALATION_KW = [
    'bomb', 'strike', 'attack', 'destroy', 'obliterate', 'war', 'military',
    'threat', 'sanction', 'blockade', 'ultimatum', 'deadline', 'missile',
    'troops', 'navy', 'carrier', 'nuclear', 'regime change', 'surrender',
    'hit harder', 'consequences', 'total destruction', 'wipe out', 'punish', 'retaliate'
]

DE_ESCALATION_KW = [
    'peace', 'talk', 'negotiate', 'deal', 'agreement', 'conversation',
    'productive', 'progress', 'ceasefire', 'extension', 'dialogue',
    'diplomatic', 'de-escalat', 'calm', 'resolve', 'cooperation',
    'good meeting', 'wonderful', 'getting along'
]

posts['iran_related'] = posts['post_text'].apply(
    lambda t: any(kw in str(t).lower() for kw in IRAN_KEYWORDS)
)

def classify_direction(text):
    text = str(text).lower()
    esc = sum(1 for kw in ESCALATION_KW if kw in text)
    deesc = sum(1 for kw in DE_ESCALATION_KW if kw in text)
    if esc > deesc and esc > 0: return 'escalation'
    elif deesc > esc and deesc > 0: return 'de-escalation'
    return 'neutral'

posts['post_direction'] = posts['post_text'].apply(classify_direction)
posts['fabrication_flag'] = False

fabrication_dates = {
    '2026-03-23': 'Productive conversations — Iran denied',
    '2026-03-09': 'War complete then hit 20x harder',
    '2026-03-06': 'Unconditional surrender — Iran did not surrender',
    '2026-03-24': '5-day extension — backed away from deadline',
}
for date_str in fabrication_dates:
    mask = (posts['date'] == pd.Timestamp(date_str)) & posts['iran_related']
    posts.loc[mask, 'fabrication_flag'] = True

iran_posts = posts[posts['iran_related']].copy()
print(f'Iran posts: {len(iran_posts)}')
print(f'  Escalation: {(iran_posts["post_direction"]=="escalation").sum()}')
print(f'  De-escalation: {(iran_posts["post_direction"]=="de-escalation").sum()}')
print(f'  Neutral: {(iran_posts["post_direction"]=="neutral").sum()}')
print(f'  Fabrication flagged: {iran_posts["fabrication_flag"].sum()}')

Iran posts: 2695
  Escalation: 1986
  De-escalation: 179
  Neutral: 530
  Fabrication flagged: 38


In [6]:
# Cell 3: Clean price data
oil = pd.merge(wti, brent, on='date', how='outer').sort_values('date').reset_index(drop=True)
oil['wti_return'] = oil['wti_close'].pct_change() * 100
oil['brent_return'] = oil['brent_close'].pct_change() * 100
oil['daily_return'] = oil['wti_return']

print(f'Oil data: {len(oil)} trading days')
print(f'  WTI: ${oil["wti_close"].min():.2f} - ${oil["wti_close"].max():.2f}')
print(f'  Brent: ${oil["brent_close"].min():.2f} - ${oil["brent_close"].max():.2f}')

Oil data: 379 trading days
  WTI: $55.44 - $98.71
  Brent: $59.93 - $118.42


In [7]:
# Cell 4: Merge datasets on date (master dataframe)

# Daily post aggregation
daily_posts = posts.groupby('date').agg(
    total_posts=('post_id', 'count'),
    iran_posts=('iran_related', 'sum'),
).reset_index()

# Iran-specific daily aggregation
iran_daily = iran_posts.groupby('date').agg(
    escalation_count=('post_direction', lambda x: (x == 'escalation').sum()),
    deescalation_count=('post_direction', lambda x: (x == 'de-escalation').sum()),
    fabrication_any=('fabrication_flag', 'any'),
    first_post_time=('timestamp', 'min'),
    post_count=('post_id', 'count'),
).reset_index()

def daily_direction(row):
    if row['escalation_count'] > row['deescalation_count']: return 'escalation'
    elif row['deescalation_count'] > row['escalation_count']: return 'de-escalation'
    elif row['escalation_count'] > 0: return 'mixed'
    return 'neutral'

iran_daily['post_direction'] = iran_daily.apply(daily_direction, axis=1)
iran_daily['fabrication_flag'] = iran_daily['fabrication_any']

# Merge
master = oil.copy()
master = pd.merge(master, daily_posts[['date', 'total_posts', 'iran_posts']],
                  on='date', how='left')
master['total_posts'] = master['total_posts'].fillna(0).astype(int)
master['iran_posts'] = master['iran_posts'].fillna(0).astype(int)

master = pd.merge(master, iran_daily[['date', 'post_direction', 'fabrication_flag',
                                       'escalation_count', 'deescalation_count',
                                       'first_post_time']],
                  on='date', how='left')
master['post_direction'] = master['post_direction'].fillna('none')
master['fabrication_flag'] = master['fabrication_flag'].fillna(False)
master['escalation_count'] = master['escalation_count'].fillna(0).astype(int)
master['deescalation_count'] = master['deescalation_count'].fillna(0).astype(int)

print(f'Master: {len(master)} rows, {len(master.columns)} columns')

Master: 379 rows, 13 columns


In [8]:
# Cell 5: Feature engineering
master['abs_return'] = master['daily_return'].abs()
master['rolling_mean_abs_return'] = master['abs_return'].rolling(20, min_periods=5).mean()
master['rolling_std_abs_return'] = master['abs_return'].rolling(20, min_periods=5).std()
master['volume_anomaly_z'] = (
    (master['abs_return'] - master['rolling_mean_abs_return']) /
    master['rolling_std_abs_return'].replace(0, np.nan)
).fillna(0)

master['is_iran_post_day'] = (master['iran_posts'] > 0).astype(int)
master['prev_day_return'] = master['daily_return'].shift(1)
master['next_day_return'] = master['daily_return'].shift(-1)

# Placeholder columns for later notebooks
master['gemini_escalation_score'] = np.nan
master['gemini_fabrication_score'] = np.nan
master['polymarket_volume'] = np.nan
master['composite_score'] = np.nan

# Filter to study period
master = master[(master['date'] >= '2024-10-01') & (master['date'] <= '2026-03-27')]
master = master.sort_values('date').reset_index(drop=True)

# Key stats
post_days = master[master['is_iran_post_day'] == 1]['abs_return']
nonpost_days = master[master['is_iran_post_day'] == 0]['abs_return']
print(f'Trading days: {len(master)}')
print(f'Iran post days: {master["is_iran_post_day"].sum()} | Non-post: {(master["is_iran_post_day"]==0).sum()}')
print(f'Post-day abs return: {post_days.mean():.3f}% vs Non-post: {nonpost_days.mean():.3f}%')
print(f'Ratio: {post_days.mean()/nonpost_days.mean():.2f}x')

Trading days: 379
Iran post days: 310 | Non-post: 69
Post-day abs return: 1.739% vs Non-post: 1.233%
Ratio: 1.41x


In [9]:
# Cell 6: Save
master.to_csv(os.path.join(PROCESSED, 'master.csv'), index=False)
iran_posts.to_csv(os.path.join(PROCESSED, 'iran_posts_cleaned.csv'), index=False)

print(f'Saved master.csv: {len(master)} rows, {len(master.columns)} columns')
print(f'Saved iran_posts_cleaned.csv: {len(iran_posts)} posts')
print(f'\nColumns: {list(master.columns)}')

Saved master.csv: 379 rows, 24 columns
Saved iran_posts_cleaned.csv: 2695 posts

Columns: ['date', 'wti_close', 'brent_close', 'wti_return', 'brent_return', 'daily_return', 'total_posts', 'iran_posts', 'post_direction', 'fabrication_flag', 'escalation_count', 'deescalation_count', 'first_post_time', 'abs_return', 'rolling_mean_abs_return', 'rolling_std_abs_return', 'volume_anomaly_z', 'is_iran_post_day', 'prev_day_return', 'next_day_return', 'gemini_escalation_score', 'gemini_fabrication_score', 'polymarket_volume', 'composite_score']
